# NB09 — Explanatory Models (M0–M3), Outcome A and Outcome B

**Authority**

```
CANONICAL_BASE_SHA      6368687c3def9786ad886d3c4886862403e22dd1   (descends 17e5a47)
AUDITED_PROTOCOL_SHA    81b682527c587f64c817b7dab74f538f16bf9152
PROTOCOL_AUDIT_SHA      33c59d58a00202e961aa68a2f9e23fcf27033004   PRE_NB09_PROTOCOL_AUDIT_PASS
AUTHORITY_ERRATUM_SHA   1dd6f753bf74dfa704c59a4b61373ec047c6ef2a   N-01 closed at 85aa19e
PROTOCOL                PRE_NB09_PROTOCOL_v001  (ssot_nb09/)
```

Nothing in the frozen contract is re-opened here: no predictor is reselected, no variable is dropped
for a VIF, `B` and the seeds are as frozen, the SE family, outcomes, cohort and model ladder are
unchanged.

**Binding interpretation restrictions** (`VD-NB09-OPERATIONALIZATION-01`)

```
SCRIPT_MIXING_BLOCK_INTERNAL_REDUNDANCY                 = YES
M3_INTERNAL_COLLINEARITY_REVIEW                         = YES
RAW_CHUNK_COUNT_COEFFICIENT_SUBSTANTIVE_INTERPRETATION  = PROHIBITED
RQ5_PRIMARY                                             = BLOCK_LEVEL_M3_MINUS_M2
source_domain_cell                                      = OBSERVED_STRATUM_CONTROL
translation_direction                                   = identified primarily within source 025
PAIR_LEVEL_MULTIPLIER_BOOTSTRAP != DUPLICATE_CLUSTER_DEPENDENCE_CORRECTION
```

Effect magnitude first. Individual coefficient p-values are secondary inferential detail and are
never used to rank predictors. No causal language anywhere.

In [1]:
from __future__ import annotations
import json, hashlib, os, time, platform
from pathlib import Path
import numpy as np, duckdb, scipy, pyarrow, pyarrow.parquet as pq
from scipy import stats
from scipy.linalg import cho_factor, cho_solve

from tokenization_premium.telemetry import RuntimeTelemetry

ROOT = Path("/home/sieg/projects-wsl/KOEN_nb09_20260818")
REG, RUNTIME = ROOT / "data/registry", ROOT / ".runtime/nb09"
RUNTIME.mkdir(parents=True, exist_ok=True)
for d in ("outputs/reports", "outputs/tables", "outputs/manifests", "docs/results/nb09"):
    (ROOT / d).mkdir(parents=True, exist_ok=True)

SHAS = {"CANONICAL_BASE_SHA": "6368687c3def9786ad886d3c4886862403e22dd1",
        "AUDITED_PROTOCOL_SHA": "81b682527c587f64c817b7dab74f538f16bf9152",
        "PROTOCOL_AUDIT_SHA": "33c59d58a00202e961aa68a2f9e23fcf27033004",
        "AUTHORITY_ERRATUM_SHA": "1dd6f753bf74dfa704c59a4b61373ec047c6ef2a"}

# 계약과 seed는 동결분에서 읽는다. 여기서 다시 타이핑하지 않는다.
CONTRACT = json.loads((ROOT / "ssot_nb09/02_NB09_MODEL_MATRIX_CONTRACT_v001.json").read_text("utf-8"))
SEEDS    = json.loads((ROOT / "ssot_nb09/03_NB09_SEED_REGISTRY_v001.json").read_text("utf-8"))
EXPECTED_N        = CONTRACT["cohort"]["N"]
EXPECTED_PAIR_SET = CONTRACT["cohort"]["pair_set_hash"]
MODELS   = {m: v["continuous"] for m, v in CONTRACT["models"].items()}
P_TARGET = {m: v["p_with_intercept"] for m, v in CONTRACT["models"].items()}
OUTCOMES = {"A": CONTRACT["outcomes"]["A"]["column"], "B": CONTRACT["outcomes"]["B"]["column"]}
B_REPS   = SEEDS["bootstrap"]["B"]
SEED     = {k: v["seed"] for k, v in SEEDS["seeds"].items()}
print("models", P_TARGET, "\noutcomes", OUTCOMES, "\nB", B_REPS, "\nseeds", SEED)

models {'M0': 8, 'M1': 23, 'M2': 27, 'M2A': 26, 'M3': 45} 
outcomes {'A': 'log_token_premium', 'B': 'log_compression_penalty'} 
B 2000 
seeds {'NB09_COEF_BOOTSTRAP_A': 1222524615, 'NB09_COEF_BOOTSTRAP_B': 1018984010, 'NB09_GLOBAL': 2703484264}


## 1. Artifact identity — fail-closed, 5/5 required

In [2]:
EXPECTED_SHA = {
 "D-01": ("PAIR_REGISTRY_v002.parquet",       "95f523d11b0e8fcfd761dee949f082e9b4590b919801441fbcfa3426010bec52"),
 "D-02": ("REP_FEATURES_v002.parquet",        "dfae8e01cd3fe2ca949d8754678e508203ad1a7aa6abea418008a33ac650d309"),
 "D-03": ("MORPH_FEATURES_KIWI_v001.parquet", "0fe5bd74e3993a7141c5c33ea78e71b2c66e3ecd296544bde2615acb43e50f7d"),
 "D-04": ("TOKEN_O200K_BASE_v001.parquet",    "1c30e3276222dd94885ae4f79fc6ab5c45e4e26226de0afd91fc6b1f7d2c16e7"),
 "D-05": ("CHUNK_O200K_BASE_v001.parquet",    "bfa98bd6cf7ee8b7254c469aed3e259ce43cc8f0529153347ca4c2c3fc1944ab")}

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h.update(b)
    return h.hexdigest()

ARTIFACTS = {}
for k, (fn, exp) in EXPECTED_SHA.items():
    got = sha256_file(REG / fn)
    ARTIFACTS[k] = {"filename": fn, "sha256": got, "expected_sha256": exp, "match": got == exp}
    print(f"{k} {fn:34s} {'MATCH' if got == exp else 'MISMATCH'}")
assert all(a["match"] for a in ARTIFACTS.values()), "G5_ARTIFACT_IDENTITY_FAIL"
print("ARTIFACT_IDENTITY = 5/5")

D-01 PAIR_REGISTRY_v002.parquet         MATCH


D-02 REP_FEATURES_v002.parquet          MATCH


D-03 MORPH_FEATURES_KIWI_v001.parquet   MATCH


D-04 TOKEN_O200K_BASE_v001.parquet      MATCH


D-05 CHUNK_O200K_BASE_v001.parquet      MATCH
ARTIFACT_IDENTITY = 5/5


## 2. Cohort materialization

The frozen derived transforms only (`PRE_NB09_PROTOCOL_v001` §3): `pair_log_size`,
`delta_whitespace_density`, `ko/en_chunk_count_log`, `source_domain_cell`.
**`ORDER BY pair_id` is asserted**, because the multiplier bootstrap consumes weights in physical
row order and reproduction depends on it.

In [3]:
P = f"read_parquet('{(REG / 'PAIR_REGISTRY_v002.parquet').as_posix()}')"
R = f"read_parquet('{(REG / 'REP_FEATURES_v002.parquet').as_posix()}')"
M = f"read_parquet('{(REG / 'MORPH_FEATURES_KIWI_v001.parquet').as_posix()}')"
T = f"read_parquet('{(REG / 'TOKEN_O200K_BASE_v001.parquet').as_posix()}')"
K = f"read_parquet('{(REG / 'CHUNK_O200K_BASE_v001.parquet').as_posix()}')"

SELECT = f'''
SELECT t.pair_id, t.log_token_premium, t.log_compression_penalty,
  t.log_code_point_ratio, t.log_byte_density_ratio,
  0.5*(ln(r.ko_codepoint_count)+ln(r.en_codepoint_count)) AS pair_log_size,
  r.ko_whitespace_density - r.en_whitespace_density        AS delta_whitespace_density,
  r.ko_latin_share, r.ko_digit_share, r.ko_punctuation_share, r.ko_symbol_other_share,
  r.en_hangul_share, r.en_digit_share, r.en_punctuation_share, r.en_symbol_other_share,
  CAST(r.ko_script_type_count   AS DOUBLE) AS ko_script_type_count,
  CAST(r.ko_script_switch_count AS DOUBLE) AS ko_script_switch_count,
  CAST(r.en_script_type_count   AS DOUBLE) AS en_script_type_count,
  CAST(r.en_script_switch_count AS DOUBLE) AS en_script_switch_count,
  m.morpheme_density, m.particle_ratio, m.ending_ratio, m.deriv_affix_ratio,
  m.function_morpheme_ratio,
  ln(k.ko_chunk_count) AS ko_chunk_count_log, ln(k.en_chunk_count) AS en_chunk_count_log,
  k.ko_mean_chunk_bytes, k.ko_p50_chunk_bytes, k.ko_p90_chunk_bytes,
  CAST(k.ko_max_chunk_bytes AS DOUBLE) AS ko_max_chunk_bytes,
  k.en_mean_chunk_bytes, k.en_p50_chunk_bytes, k.en_p90_chunk_bytes,
  CAST(k.en_max_chunk_bytes AS DOUBLE) AS en_max_chunk_bytes,
  CAST(k.ko_max_tokens_per_chunk AS DOUBLE) AS ko_max_tokens_per_chunk,
  CAST(k.en_max_tokens_per_chunk AS DOUBLE) AS en_max_tokens_per_chunk,
  k.ko_chunk_type_share_number, k.ko_chunk_type_share_punctuation, k.ko_chunk_type_share_whitespace,
  k.en_chunk_type_share_number, k.en_chunk_type_share_punctuation, k.en_chunk_type_share_whitespace,
  p.source_id || '-' || p.domain AS source_domain_cell, p.translation_direction
FROM {T} t JOIN {P} p ON p.pair_id=t.pair_id JOIN {R} r ON r.pair_id=t.pair_id
           JOIN {M} m ON m.pair_id=t.pair_id JOIN {K} k ON k.pair_id=t.pair_id
ORDER BY t.pair_id
'''

MATRIX = (RUNTIME / "nb09_matrix.parquet").as_posix()
con = duckdb.connect()
con.execute("PRAGMA memory_limit='3GB'"); con.execute("PRAGMA threads=4")
con.execute(f"PRAGMA temp_directory='{(RUNTIME/'spill').as_posix()}'")

with RuntimeTelemetry(run_id="NB09_MATERIALIZE", stage="MATERIALIZE", total=EXPECTED_N) as tel:
    con.execute(f"COPY ({SELECT}) TO '{MATRIX}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    tel.set_stage("VERIFY")
    A = f"read_parquet('{MATRIX}')"
    n_rows, n_dist = con.execute(f"SELECT count(*), count(DISTINCT pair_id) FROM {A}").fetchone()
    pair_set = con.execute(f"SELECT md5(string_agg(pair_id,'' ORDER BY pair_id)) FROM {A}").fetchone()[0]
    ordered = con.execute(
        f"SELECT count(*) FROM (SELECT pair_id, lag(pair_id) OVER () AS prev FROM {A}) "
        "WHERE prev IS NOT NULL AND pair_id < prev").fetchone()[0]
    tel.update(n_rows)
# summary는 with를 벗어난 뒤에 읽는다 — __exit__가 final_status를 COMPLETED로 확정한 다음이다.
TEL_MAT = {k: v for k, v in tel.summary().items() if k != "samples"}

assert n_rows == EXPECTED_N, f"COHORT_N_MISMATCH {n_rows}"
assert n_dist == n_rows, "PAIR_ID_NOT_UNIQUE"
assert pair_set == EXPECTED_PAIR_SET, "PAIR_SET_HASH_MISMATCH"
assert ordered == 0, "ROW_ORDER_NOT_ASCENDING_BY_PAIR_ID — HARD STOP"
print(f"N={n_rows}  pair-set={pair_set}  ORDER_BY_PAIR_ID_ASSERTED=True")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

N=3835988  pair-set=d9660d654ee449e4d0c23a0070225274  ORDER_BY_PAIR_ID_ASSERTED=True


## 3. Design construction and pass 1 — exact sufficient statistics

The full-cohort wide matrix is never held in Pandas. Two streaming passes accumulate everything:
pass 1 builds the Gram `XᵀX`, `Xᵀy` for both outcomes and `yᵀy`; pass 2 builds the HC1 meat matrix
and the bootstrap statistics. Columns are scale-normalized before the linear solve — the raw-coding
condition number is large only because the natural scales differ by orders of magnitude, and scaling
recovers the well-conditioned problem G5 measured.

In [4]:
LEVELS = {k: v for k, v in CONTRACT["reference_levels"].items()}
cell_counts = dict(con.execute(f"SELECT source_domain_cell, count(*) FROM {A} GROUP BY 1").fetchall())
dir_counts  = dict(con.execute(f"SELECT translation_direction, count(*) FROM {A} GROUP BY 1").fetchall())
CELL_REF, DIR_REF = LEVELS["source_domain_cell"], LEVELS["translation_direction"]
assert CELL_REF in cell_counts and DIR_REF in dir_counts, "REFERENCE_LEVEL_ABSENT"
CELL_LV = [l for l in sorted(cell_counts, key=lambda x: (-cell_counts[x], x)) if l != CELL_REF]
DIR_LV  = [l for l in sorted(dir_counts,  key=lambda x: (-dir_counts[x],  x)) if l != DIR_REF]
DUMMIES = [f"cell::{l}" for l in CELL_LV] + [f"dir::{l}" for l in DIR_LV]

ALL_CONT = list(dict.fromkeys(sum(MODELS.values(), [])))
DESIGN   = ["__intercept__"] + ALL_CONT + DUMMIES
IDX      = {c: i for i, c in enumerate(DESIGN)}
PCOLS    = {m: [IDX["__intercept__"]] + [IDX[c] for c in cont] + [IDX[d] for d in DUMMIES]
            for m, cont in MODELS.items()}
for m, cols in PCOLS.items():
    assert len(cols) == P_TARGET[m], f"P_MISMATCH {m}: {len(cols)} != {P_TARGET[m]}"
print("p per model:", {m: len(c) for m, c in PCOLS.items()}, "| superset", len(DESIGN))

READ = ALL_CONT + list(OUTCOMES.values()) + ["source_domain_cell", "translation_direction"]
PF = pq.ParquetFile(MATRIX)

def batches(batch_size):
    for b in PF.iter_batches(batch_size=batch_size, columns=READ):
        d = b.to_pydict(); m = len(d[ALL_CONT[0]])
        X = np.empty((m, len(DESIGN)))
        X[:, 0] = 1.0
        for c in ALL_CONT:
            X[:, IDX[c]] = np.asarray(d[c], dtype=np.float64)
        cell = np.asarray(d["source_domain_cell"], dtype=object)
        dr   = np.asarray(d["translation_direction"], dtype=object)
        for l in CELL_LV: X[:, IDX[f"cell::{l}"]] = (cell == l)
        for l in DIR_LV:  X[:, IDX[f"dir::{l}"]]  = (dr == l)
        Y = {k: np.asarray(d[v], dtype=np.float64) for k, v in OUTCOMES.items()}
        assert all(np.isfinite(Y[k]).all() for k in Y), "NONFINITE_OUTCOME"
        assert np.isfinite(X).all(), "NONFINITE_DESIGN"
        yield X, Y

pdim = len(DESIGN)
G = np.zeros((pdim, pdim)); Xty = {k: np.zeros(pdim) for k in OUTCOMES}
yty = {k: 0.0 for k in OUTCOMES}; ysum = {k: 0.0 for k in OUTCOMES}; n = 0
with RuntimeTelemetry(run_id="NB09_PASS1_GRAM", stage="GRAM", total=EXPECTED_N) as tel:
    for X, Y in batches(200_000):
        G += X.T @ X
        for k in OUTCOMES:
            Xty[k] += X.T @ Y[k]; yty[k] += float(Y[k] @ Y[k]); ysum[k] += float(Y[k].sum())
        n += X.shape[0]; tel.update(X.shape[0])
TEL_P1 = {k: v for k, v in tel.summary().items() if k != "samples"}
assert n == EXPECTED_N
SCALE = np.sqrt(np.diag(G) / n); SCALE[SCALE == 0] = 1.0
print("pass 1 done; n =", n)

p per model: {'M0': 8, 'M1': 23, 'M2': 27, 'M2A': 26, 'M3': 45} | superset 46


pass 1 done; n = 3835988


## 4. Fit — closed-form OLS, rank confirmation, R², adjusted R², AIC, BIC

In [5]:
def fit(model, out):
    cols = PCOLS[model]; p = len(cols)
    s  = SCALE[cols]
    Gs = G[np.ix_(cols, cols)] / np.outer(s, s)
    ev = np.linalg.eigvalsh(Gs)
    rank = int((ev > max(n, p) * np.finfo(float).eps * ev[-1]).sum())
    assert rank == p, f"RANK_DEFICIENT {model}: {rank}/{p}"
    c = cho_factor(Gs, lower=True)                      # 성공 자체가 양정치성의 증거다
    beta_s = cho_solve(c, Xty[out][cols] / s)
    beta   = beta_s / s
    Gb     = Gs @ beta_s
    ssr    = float(yty[out] - 2 * beta_s @ (Xty[out][cols] / s) + beta_s @ Gb)
    sst    = float(yty[out] - n * (ysum[out] / n) ** 2)
    r2     = 1.0 - ssr / sst
    adj    = 1.0 - (1.0 - r2) * (n - 1) / (n - p)
    k      = p + 1                                       # + sigma^2
    logL   = -0.5 * n * (np.log(2 * np.pi) + np.log(ssr / n) + 1.0)
    return {"model": model, "outcome": out, "n": n, "p": p, "rank": rank, "cols": cols,
            "scale": s, "chol": c, "beta": beta, "beta_s": beta_s, "ssr": ssr, "sst": sst,
            "r2": r2, "adj_r2": adj, "logL": float(logL),
            "aic": float(-2 * logL + 2 * k), "bic": float(-2 * logL + k * np.log(n)),
            "yhat_sq": float(beta_s @ Gb)}

FITS = {(m, o): fit(m, o) for o in OUTCOMES for m in MODELS}
for o in OUTCOMES:
    print(f"--- Outcome {o} ({OUTCOMES[o]}) ---")
    for m in MODELS:
        f = FITS[(m, o)]
        print(f"  {m:4s} p={f['p']:3d} rank={f['rank']:3d} R2={f['r2']:.6f} "
              f"adjR2={f['adj_r2']:.6f} AIC={f['aic']:.1f} BIC={f['bic']:.1f}")

--- Outcome A (log_token_premium) ---
  M0   p=  8 rank=  8 R2=0.028625 adjR2=0.028623 AIC=-768948.1 BIC=-768829.7
  M1   p= 23 rank= 23 R2=0.649578 adjR2=0.649576 AIC=-4679998.2 BIC=-4679682.4
  M2   p= 27 rank= 27 R2=0.656169 adjR2=0.656166 AIC=-4752819.4 BIC=-4752450.9
  M2A  p= 26 rank= 26 R2=0.655423 adjR2=0.655421 AIC=-4744511.1 BIC=-4744155.8
  M3   p= 45 rank= 45 R2=0.806939 adjR2=0.806936 AIC=-6966697.3 BIC=-6966092.0
--- Outcome B (log_compression_penalty) ---
  M0   p=  8 rank=  8 R2=0.182000 adjR2=0.181999 AIC=-2720483.3 BIC=-2720364.8
  M1   p= 23 rank= 23 R2=0.509203 adjR2=0.509200 AIC=-4679998.2 BIC=-4679682.4
  M2   p= 27 rank= 27 R2=0.518433 adjR2=0.518430 AIC=-4752819.4 BIC=-4752450.9
  M2A  p= 26 rank= 26 R2=0.517389 adjR2=0.517386 AIC=-4744511.1 BIC=-4744155.8
  M3   p= 45 rank= 45 R2=0.729600 adjR2=0.729597 AIC=-6966697.3 BIC=-6966092.0


## 5. Pass 2 — HC1 meat matrix and the multiplier bootstrap

`HC1_SCALE = N / (N − p)` is recorded literally per model, per audit review `R-04`.

The bootstrap is the wild/multiplier form with Rademacher weights: `y* = ŷ + e·w`, so
`β* − β = G⁻¹ Xᵀ(e∘w)`. Only `s_b = Xᵀ(e∘w)` and `u_b = Σ ŷ e w` need accumulating, which keeps the
whole procedure streaming at `B = 2000`.

In [6]:
BOOT_CHUNK = 20_000
rng = {o: np.random.default_rng(SEED[f"NB09_COEF_BOOTSTRAP_{o}"]) for o in OUTCOMES}
meat = {key: np.zeros((f["p"], f["p"])) for key, f in FITS.items()}
sb   = {key: np.zeros((f["p"], B_REPS)) for key, f in FITS.items()}
ub   = {key: np.zeros(B_REPS) for key in FITS}
seen = 0
with RuntimeTelemetry(run_id="NB09_PASS2_HC1_BOOT", stage="HC1_AND_BOOTSTRAP",
                      total=EXPECTED_N) as tel:
    for X, Y in batches(BOOT_CHUNK):
        m = X.shape[0]
        W = {o: rng[o].integers(0, 2, size=(m, B_REPS)).astype(np.float64) * 2.0 - 1.0
             for o in OUTCOMES}
        for (mod, o), f in FITS.items():
            Xc = X[:, f["cols"]]
            yhat = Xc @ f["beta"]
            e = Y[o] - yhat
            Z = Xc * e[:, None]
            meat[(mod, o)] += Z.T @ Z
            sb[(mod, o)]   += Z.T @ W[o]
            ub[(mod, o)]   += (yhat * e) @ W[o]
        seen += m; tel.update(m)
TEL_P2 = {k: v for k, v in tel.summary().items() if k != "samples"}
assert seen == EXPECTED_N, f"PASS2_ROW_DRIFT {seen}"
print("pass 2 done; rows =", seen)

pass 2 done; rows = 3835988


In [7]:
HC1_SCALE = {}
for key, f in FITS.items():
    p, s = f["p"], f["scale"]
    hc1 = n / (n - p)
    HC1_SCALE[key] = hc1
    Ms = meat[key] / np.outer(s, s)
    Gi_Ms = cho_solve(f["chol"], Ms)
    Vs = cho_solve(f["chol"], Gi_Ms.T).T * hc1
    V = Vs / np.outer(s, s)
    f["hc1_scale"] = float(hc1)
    f["V"] = V
    f["se"] = np.sqrt(np.diag(V))
    f["t"] = f["beta"] / f["se"]
    f["p_value"] = 2.0 * stats.norm.sf(np.abs(f["t"]))       # secondary detail only
    f["ci_lo"] = f["beta"] - 1.959963984540054 * f["se"]
    f["ci_hi"] = f["beta"] + 1.959963984540054 * f["se"]
    # bootstrap coefficient dispersion + replicate R^2
    dB = cho_solve(f["chol"], sb[key] / s[:, None]) / s[:, None]   # G^-1 (X'(e*w)) per replicate
    f["boot_coef_sd"] = dB.std(axis=1, ddof=1)
    yy_star = f["yhat_sq"] + 2.0 * ub[key] + f["ssr"]
    Xty_star = (G[np.ix_(f["cols"], f["cols"])] @ f["beta"])[:, None] + sb[key]
    beta_star = f["beta"][:, None] + dB
    ssr_star = yy_star - np.einsum("ij,ij->j", beta_star, Xty_star)
    mean_star = (ysum[f["outcome"]] + sb[key][0]) / n
    sst_star = yy_star - n * mean_star ** 2
    f["r2_boot"] = 1.0 - ssr_star / sst_star
print("HC1_SCALE:", {f"{k[0]}/{k[1]}": round(v, 12) for k, v in HC1_SCALE.items()})

HC1_SCALE: {'M0/A': 1.000002085517, 'M1/A': 1.000005995884, 'M2/A': 1.000007038653, 'M2A/A': 1.000006777961, 'M3/A': 1.000011731144, 'M0/B': 1.000002085517, 'M1/B': 1.000005995884, 'M2/B': 1.000007038653, 'M2A/B': 1.000006777961, 'M3/B': 1.000011731144}


## 6. Primary nested block comparisons

`RQ3 = M1 − M0`, `RQ4 = M2 − M1`, `RQ5 = M3 − M2`. `M2A − M1` is the pre-specified morphology
sensitivity, never a replacement primary. Effect magnitude is `ΔR²` and partial `R²`; the nested
statistic and its p-value are reported beside them and never as the finding — at `N ≈ 3.8M` nested
tests may have very high power and can reject practically negligible increments.

In [8]:
COMPARISONS = [("RQ3", "M1", "M0"), ("RQ4", "M2", "M1"), ("RQ5", "M3", "M2"),
               ("SENS_M2A", "M2A", "M1")]
BLOCKS = []
for o in OUTCOMES:
    for rq, full, red in COMPARISONS:
        ff, fr = FITS[(full, o)], FITS[(red, o)]
        q = ff["p"] - fr["p"]
        d_r2 = ff["r2"] - fr["r2"]
        partial = d_r2 / (1.0 - fr["r2"])
        lr = n * np.log(fr["ssr"] / ff["ssr"])                      # protocol §5, nested only
        lr_p = float(stats.chi2.sf(lr, q))
        added = [c for c in ff["cols"] if c not in set(fr["cols"])]
        pos = [ff["cols"].index(c) for c in added]
        delta = ff["beta"][pos]
        Vsub = ff["V"][np.ix_(pos, pos)]
        wald = float(delta @ np.linalg.solve(Vsub, delta))          # HC1-robust companion
        wald_p = float(stats.chi2.sf(wald, len(pos)))
        db = ff["r2_boot"] - fr["r2_boot"]
        lo, hi = np.percentile(db, [2.5, 97.5])
        BLOCKS.append({"outcome": o, "comparison": rq, "full": full, "reduced": red,
            "df": int(q), "n": n, "r2_full": ff["r2"], "r2_reduced": fr["r2"],
            "delta_r2": float(d_r2), "partial_r2": float(partial),
            "adj_r2_full": ff["adj_r2"], "adj_r2_reduced": fr["adj_r2"],
            "aic_full": ff["aic"], "aic_reduced": fr["aic"],
            "bic_full": ff["bic"], "bic_reduced": fr["bic"],
            "nested_statistic": "likelihood_ratio_chi2", "nested_stat_value": float(lr),
            "nested_p_value": lr_p,
            "hc1_robust_wald_chi2": wald, "hc1_robust_wald_p_value": wald_p,
            "delta_r2_bootstrap_ci95": [float(lo), float(hi)],
            "delta_r2_bootstrap_sd": float(db.std(ddof=1)),
            "bootstrap_B": B_REPS, "primary_evidence": "delta_r2 / partial_r2 / stability"})
        print(f"{o} {rq:9s} {red}->{full:4s} dR2={d_r2:.6f} partial={partial:.6f} "
              f"boot95=[{lo:.6f},{hi:.6f}] LRchi2={lr:.1f} df={q}")

A RQ3       M0->M1   dR2=0.620954 partial=0.639252 boot95=[0.620245,0.621652] LRchi2=3911080.1 df=15
A RQ4       M1->M2   dR2=0.006590 partial=0.018807 boot95=[0.006476,0.006701] LRchi2=72829.2 df=4
A RQ5       M2->M3   dR2=0.150770 partial=0.438500 boot95=[0.150248,0.151317] LRchi2=2213913.9 df=18
A SENS_M2A  M1->M2A  dR2=0.005845 partial=0.016679 boot95=[0.005744,0.005944] LRchi2=64518.9 df=3
B RQ3       M0->M1   dR2=0.327203 partial=0.400004 boot95=[0.326135,0.328256] LRchi2=1959545.0 df=15
B RQ4       M1->M2   dR2=0.009230 partial=0.018807 boot95=[0.009046,0.009414] LRchi2=72829.2 df=4
B RQ5       M2->M3   dR2=0.211167 partial=0.438500 boot95=[0.210306,0.212063] LRchi2=2213913.9 df=18
B SENS_M2A  M1->M2A  dR2=0.008186 partial=0.016679 boot95=[0.008030,0.008346] LRchi2=64518.9 df=3


## 6b. Structural relationship between Outcome A and Outcome B — measured, not assumed

`Y_B = Y_A − logCR − logBDR` by the exact decomposition, and `logCR`/`logBDR` are **columns of M1**
under the approved common ladder (`SPEC-02`). Adding a linear combination of design columns to the
outcome leaves the OLS residual unchanged, so for every model that contains both components the
Outcome B fit must be the Outcome A fit with those two coefficients shifted by exactly `−1`.

This is verified here rather than asserted, because it determines how the two outcomes may be read.

In [9]:
AB = {}
for m in MODELS:
    fa, fb = FITS[(m, "A")], FITS[(m, "B")]
    terms = [DESIGN[c] for c in fa["cols"]]
    d = fa["beta"] - fb["beta"]
    comp = {t: float(d[i]) for i, t in enumerate(terms)
            if t in ("log_code_point_ratio", "log_byte_density_ratio")}
    other = float(max((abs(d[i]) for i, t in enumerate(terms) if t not in comp), default=0.0))
    contains = len(comp) == 2
    AB[m] = {"model": m, "contains_both_decomposition_components": contains,
             "ssr_A": fa["ssr"], "ssr_B": fb["ssr"],
             "ssr_relative_difference": abs(fa["ssr"] - fb["ssr"]) / fa["ssr"],
             "coef_difference_on_components": comp,
             "max_abs_coef_difference_elsewhere": other,
             "aic_A": fa["aic"], "aic_B": fb["aic"],
             "sst_A": fa["sst"], "sst_B": fb["sst"]}
    print(f"{m:4s} contains_components={contains}  SSR rel.diff={AB[m]['ssr_relative_difference']:.3e}"
          f"  coef diff on components={ {k: round(v, 12) for k, v in comp.items()} }"
          f"  max|diff| elsewhere={other:.3e}")
    if contains:
        assert AB[m]["ssr_relative_difference"] < 1e-9, f"AB_RESIDUAL_INVARIANCE_VIOLATED {m}"
        assert all(abs(v - 1.0) < 1e-8 for v in comp.values()), f"AB_SHIFT_NOT_UNIT {m}"
        assert other < 1e-6, f"AB_OTHER_COEFS_MOVED {m}"

AB_FINDING = {
 "id": "B-N01",
 "class": "STRUCTURAL_CONSEQUENCE_OF_THE_APPROVED_COMMON_LADDER",
 "statement": ("For every model containing both log_code_point_ratio and log_byte_density_ratio "
               "(M1, M2, M2A, M3), the Outcome B fit is the Outcome A fit with those two "
               "coefficients shifted by exactly -1 and every other coefficient unchanged. The "
               "residual vector, and therefore SSR, AIC, BIC and every nested test statistic, are "
               "identical between the two outcomes."),
 "measured": {"unit_shift_on_components": True, "max_abs_other_coef_difference":
              max(AB[m]["max_abs_coef_difference_elsewhere"] for m in MODELS if
                  AB[m]["contains_both_decomposition_components"]),
              "max_ssr_relative_difference":
              max(AB[m]["ssr_relative_difference"] for m in MODELS if
                  AB[m]["contains_both_decomposition_components"])},
 "consequences": [
   "RQ4 and RQ5 partial R-squared are IDENTICAL across the two outcomes by construction; "
   "they are not two independent confirmations.",
   "Delta R-squared differs between outcomes only through the SST denominator, because the "
   "numerator SSR difference is the same quantity.",
   "AIC and BIC coincide for M1, M2, M2A and M3; only M0 differs, because M0 omits the two "
   "decomposition components.",
   "Only RQ3 (M1 - M0) carries genuinely different information between the outcomes."],
 "relation_to_authority": (
   "SSOT 18.2's illustrative Outcome B form omits logCR and logBDR; PRE_NB09_PROTOCOL_v001 "
   "SPEC-02 recorded that the approved common ladder includes them and flagged the divergence "
   "for NB09 interpretation. This is the measured consequence of that divergence. No protocol "
   "change is made here - the protocol is audited and closed, and it was executed as frozen."),
 "director_attention": (
   "Whether Outcome B should be re-specified without the two representation ratios, so that it "
   "carries information M1 does not already contain, is a decision for the Director. Registered "
   "as an NB11 sensitivity candidate; NOT actioned here."),
 "not_a_defect": ("This is not an implementation error and not outcome leakage: neither outcome is "
                  "reconstructible from its own right-hand side, which G5 verified and this run "
                  "re-confirms through full rank at every model.")}
print("\n" + AB_FINDING["statement"])

M0   contains_components=False  SSR rel.diff=3.987e-01  coef diff on components={}  max|diff| elsewhere=2.907e-01
M1   contains_components=True  SSR rel.diff=1.931e-14  coef diff on components={'log_code_point_ratio': 1.0, 'log_byte_density_ratio': 1.000000000002}  max|diff| elsewhere=2.803e-12
M2   contains_components=True  SSR rel.diff=1.731e-13  coef diff on components={'log_code_point_ratio': 1.0, 'log_byte_density_ratio': 1.000000000003}  max|diff| elsewhere=4.005e-12
M2A  contains_components=True  SSR rel.diff=4.196e-13  coef diff on components={'log_code_point_ratio': 1.0, 'log_byte_density_ratio': 1.000000000003}  max|diff| elsewhere=3.752e-12
M3   contains_components=True  SSR rel.diff=1.091e-13  coef diff on components={'log_code_point_ratio': 1.0, 'log_byte_density_ratio': 1.000000000005}  max|diff| elsewhere=8.281e-12

For every model containing both log_code_point_ratio and log_byte_density_ratio (M1, M2, M2A, M3), the Outcome B fit is the Outcome A fit with those two coef

## 7. Persist artifacts

In [10]:
import csv, subprocess
CODE_SHA = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"],
                          capture_output=True, text=True).stdout.strip()
RESTRICTIONS = {
    "SCRIPT_MIXING_BLOCK_INTERNAL_REDUNDANCY": "YES",
    "M3_INTERNAL_COLLINEARITY_REVIEW": "YES",
    "RAW_CHUNK_COUNT_COEFFICIENT_SUBSTANTIVE_INTERPRETATION": "PROHIBITED",
    "RQ5_PRIMARY": "BLOCK_LEVEL_M3_MINUS_M2",
    "source_domain_cell": "OBSERVED_STRATUM_CONTROL — no pure source effect, no pure domain effect",
    "translation_direction": "identified primarily from within-025 support; 026 has no EN_TO_KO",
    "PAIR_LEVEL_MULTIPLIER_BOOTSTRAP": "!= DUPLICATE_CLUSTER_DEPENDENCE_CORRECTION",
    "coefficient_p_values": "secondary inferential detail; never used to rank predictors",
    "causal_language": "PROHIBITED",
    "outcome_A_and_B_narratives": "never merged; B is not the cause of A; the decomposition is an "
                                  "accounting identity, not causal mediation"}

model_rows, coef_rows = [], []
for (mod, o), f in FITS.items():
    model_rows.append({"outcome": o, "outcome_column": OUTCOMES[o], "model": mod, "n": f["n"],
        "p": f["p"], "rank": f["rank"], "r2": f["r2"], "adj_r2": f["adj_r2"],
        "aic": f["aic"], "bic": f["bic"], "ssr": f["ssr"], "sst": f["sst"],
        "hc1_scale": f["hc1_scale"], "se_type": "HC1"})
    for j, ci in enumerate(f["cols"]):
        coef_rows.append({"outcome": o, "model": mod, "term": DESIGN[ci],
            "coef": float(f["beta"][j]), "hc1_se": float(f["se"][j]),
            "ci95_lo": float(f["ci_lo"][j]), "ci95_hi": float(f["ci_hi"][j]),
            "z": float(f["t"][j]), "p_value_secondary": float(f["p_value"][j]),
            "bootstrap_sd": float(f["boot_coef_sd"][j])})

RESULTS = {"artifact_id": "NB09_EXPLANATORY_RESULTS_v001", **SHAS,
    "protocol_id": "PRE_NB09_PROTOCOL_v001", "code_sha": CODE_SHA,
    "cohort": {"N": n, "pair_set_hash": pair_set, "order_by_pair_id_asserted": True},
    "artifacts": ARTIFACTS, "outcomes": OUTCOMES,
    "estimator": "fixed-effects OLS, closed form from exact streaming Gram",
    "se_type": "HC1", "hc1_scale_formula": "N / (N - p)",
    "hc1_scale": {f"{k[0]}/{k[1]}": v["hc1_scale"] for k, v in FITS.items()},
    "bootstrap": {"method": "pair-level multiplier (wild), Rademacher", "B": B_REPS,
                  "seeds": SEED, "row_order": "ORDER BY pair_id (asserted)"},
    "models": model_rows, "block_comparisons": BLOCKS,
    "outcome_ab_structural_relationship": {"per_model": AB, "finding": AB_FINDING},
    "claim_restrictions": RESTRICTIONS,
    "software": {"python": platform.python_version(), "numpy": np.__version__,
                 "scipy": scipy.__version__, "duckdb": duckdb.__version__,
                 "pyarrow": pyarrow.__version__}}
DIAG = {"artifact_id": "NB09_MODEL_DIAGNOSTICS_v001", **SHAS,
    "rank_by_model": {f"{k[0]}/{k[1]}": {"p": v["p"], "rank": v["rank"],
                                         "full_rank": v["rank"] == v["p"]}
                      for k, v in FITS.items()},
    "hc1_scale": {f"{k[0]}/{k[1]}": v["hc1_scale"] for k, v in FITS.items()},
    "coefficients": coef_rows,
    "runtime_telemetry": {"materialize": TEL_MAT, "pass1_gram": TEL_P1, "pass2_hc1_boot": TEL_P2},
    "claim_restrictions": RESTRICTIONS}
MANIFEST = {"artifact_id": "NB09_RUN_MANIFEST_v001", **SHAS, "code_sha": CODE_SHA,
    "protocol_id": "PRE_NB09_PROTOCOL_v001", "notebook": "notebooks/09_explanatory_models.ipynb",
    "N": n, "pair_set_hash": pair_set,
    "artifact_sha256": {k: v["sha256"] for k, v in ARTIFACTS.items()},
    "artifact_identity": f"{sum(a['match'] for a in ARTIFACTS.values())}/5",
    "se_type": "HC1", "hc1_scale_formula": "N / (N - p)",
    "hc1_scale": {f"{k[0]}/{k[1]}": v["hc1_scale"] for k, v in FITS.items()},
    "bootstrap_B": B_REPS, "bootstrap_seeds": SEED,
    "order_by_pair_id_asserted": True,
    "runtime_sec": {"materialize": TEL_MAT.get("elapsed_sec"),
                    "pass1": TEL_P1.get("elapsed_sec"), "pass2": TEL_P2.get("elapsed_sec")},
    "eng_obs_001_note": ("ENG-OBS-001 binds stages exceeding 30 s. materialize ran ~10 s and is "
                         "below that threshold, so its sample count is 2 by design, not by defect; "
                         "pass1 and pass2 both exceed 30 s and carry periodic 10 s sampling."),
    "eng_obs_001": {k: {"interval_sec": v.get("interval_sec"),
                        "sample_count": v.get("sample_count"),
                        "r1_periodic_sampling": v.get("r1_periodic_sampling"),
                        "min_mem_available_gib": v.get("min_mem_available_gib"),
                        "peak_rss_gib": v.get("peak_rss_gib"),
                        "worst_memory_status": v.get("worst_memory_status"),
                        "red_or_worse_sample_count": v.get("red_or_worse_sample_count"),
                        "final_status": v.get("final_status")}
                    for k, v in (("materialize", TEL_MAT), ("pass1", TEL_P1), ("pass2", TEL_P2))},
    "claim_restrictions": RESTRICTIONS, "raw_text_persisted": False}

def dump(rel, obj):
    p = ROOT / rel
    p.write_text(json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False) + "\n", "utf-8")
    print("wrote", rel)

dump("outputs/reports/NB09_EXPLANATORY_RESULTS_v001.json", RESULTS)
dump("outputs/reports/NB09_MODEL_DIAGNOSTICS_v001.json", DIAG)
dump("outputs/manifests/NB09_RUN_MANIFEST_v001.json", MANIFEST)

def csv_write(rel, rows):
    p = ROOT / rel
    with open(p, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    print("wrote", rel)

csv_write("outputs/tables/NB09_MODEL_SUMMARY_v001.csv", model_rows)
csv_write("outputs/tables/NB09_BLOCK_COMPARISONS_v001.csv", BLOCKS)

wrote outputs/reports/NB09_EXPLANATORY_RESULTS_v001.json
wrote outputs/reports/NB09_MODEL_DIAGNOSTICS_v001.json
wrote outputs/manifests/NB09_RUN_MANIFEST_v001.json
wrote outputs/tables/NB09_MODEL_SUMMARY_v001.csv
wrote outputs/tables/NB09_BLOCK_COMPARISONS_v001.csv


## 8. Reading of record

Every statement below is a **conditional association** on this cohort under the fixed `o200k_base`
raw-text Track A configuration. Nothing here is causal, and nothing generalizes to another
tokenizer, corpus or language.

**Outcome A** — total tokenization premium association.
**Outcome B** — compression asymmetry after representation accounting. Outcome B is **not** "the
cause of" Outcome A; the exact decomposition is an accounting identity, not causal mediation.

**Restrictions enforced in every table above and in every persisted artifact**

- `SCRIPT_MIXING_BLOCK_INTERNAL_REDUNDANCY = YES` — `script_type_count` and `script_switch_count`
  are not two independent substantive linguistic effects. Their construct definitions make
  `1{switch ≥ 1}` identically `1{type ≥ 2}`, so only the block-level contribution is read.
- `M3_INTERNAL_COLLINEARITY_REVIEW = YES`,
  `RAW_CHUNK_COUNT_COEFFICIENT_SUBSTANTIVE_INTERPRETATION = PROHIBITED`,
  `RQ5_PRIMARY = BLOCK_LEVEL_M3_MINUS_M2`.
- `source_domain_cell` is an observed-stratum control — no pure source effect, no pure domain effect.
- `translation_direction` is identified primarily from within-source-025 support; 026 contains no
  `EN_TO_KO`.
- HC1 standard errors do not account for dependence between paraphrase or near-duplicate pairs. The
  multiplier bootstrap does not correct it either. That remains an NB09 limitation, NB11 candidate
  #12, and a PRE-NB10 grouping prerequisite.

In [11]:
lines = ["# NB09 — Explanatory Models", "",
         "Conditional associations only. No causal claim. `o200k_base`, Track A raw text.", "",
         "```", f"N = {n:,}   pair-set {pair_set}",
         f"estimator  fixed-effects OLS   SE  HC1 (scale = N/(N-p))",
         f"bootstrap  wild/Rademacher B={B_REPS}  seeds {SEED}", "```", "",
         "## Model fit", "",
         "| outcome | model | p | rank | R² | adj R² | AIC | BIC | HC1 scale |",
         "|---|---|---:|---:|---:|---:|---:|---:|---:|"]
for r in model_rows:
    lines.append(f"| {r['outcome']} | {r['model']} | {r['p']} | {r['rank']} | {r['r2']:.6f} | "
                 f"{r['adj_r2']:.6f} | {r['aic']:.1f} | {r['bic']:.1f} | {r['hc1_scale']:.9f} |")
lines += ["", "## Primary block comparisons", "",
          "| outcome | comparison | ΔR² | partial R² | bootstrap 95% CI | df | LR χ² |",
          "|---|---|---:|---:|---|---:|---:|"]
for b in BLOCKS:
    ci = b["delta_r2_bootstrap_ci95"]
    lines.append(f"| {b['outcome']} | {b['comparison']} ({b['reduced']}→{b['full']}) | "
                 f"{b['delta_r2']:.6f} | {b['partial_r2']:.6f} | "
                 f"[{ci[0]:.6f}, {ci[1]:.6f}] | {b['df']} | {b['nested_stat_value']:.1f} |")
lines += ["", "## B-N01 — Outcome A / Outcome B are the same fit above M0", "",
          AB_FINDING["statement"], ""] + \
         [f"- {c}" for c in AB_FINDING["consequences"]] + \
         ["", AB_FINDING["relation_to_authority"], "",
          AB_FINDING["director_attention"], "", AB_FINDING["not_a_defect"], ""]
lines += ["", "## Binding restrictions", "", "```"] + \
         [f"{k} = {v}" for k, v in RESTRICTIONS.items()] + ["```", ""]
(ROOT / "docs/results/nb09/README.md").write_text("\n".join(lines) + "\n", "utf-8")
print("wrote docs/results/nb09/README.md")
print("NB09_EXPLANATORY_EXECUTION_COMPLETE")

wrote docs/results/nb09/README.md
NB09_EXPLANATORY_EXECUTION_COMPLETE
